In [1]:
import os
os.environ["GROQ_API_KEY"]="gsk_gVT"

In [7]:
!pip install -q youtube-transcript-api langchain-community langchain-groq faiss-cpu tiktoken python-dotenv

In [9]:
!pip install -q langchain-huggingface sentence-transformers

In [10]:
from youtube_transcript_api import YouTubeTranscriptApi, TranscriptsDisabled

from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_community.vectorstores import FAISS

from langchain_groq import ChatGroq

from langchain_core.prompts import PromptTemplate

from langchain_huggingface import HuggingFaceEmbeddings

In [11]:
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [13]:
#indexing
video_id = "LPZh9BOjkQs"  # sirf video ID, full URL nahi

try:
    transcript_list = YouTubeTranscriptApi().fetch(
        video_id,
        languages=["en"]
    )

    transcript = " ".join(chunk.text for chunk in transcript_list)

    print(transcript)

except TranscriptsDisabled:
    print("Transcript is disabled")

[Submit subtitle corrections at criblate.com]
Imagine you happen across a short movie script that describes a scene between a person and their AI assistant. The script has what the person asks the AI, but the AI's response has been torn off. Suppose you also have this powerful magical machine that can take any text and provide a sensible prediction of what word comes next. You could then finish the script by feeding in what you have to the machine, seeing what it would predict to start the AI's answer, and then repeating this over and over with a growing script completing the dialogue. When you interact with a chatbot, this is exactly what's happening. A large language model is a sophisticated mathematical function that predicts what word comes next for any piece of text. Instead of predicting one word with certainty, though, what it does is assign a probability to all possible next words. To build a chatbot, what you do is lay out some text that describes an interaction between a user

In [16]:
#splitting the document

splitter=RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=200)
chunnks=splitter.create_documents([transcript])

In [17]:
len(chunnks)

11

In [18]:
chunnks[10]

Document(metadata={}, page_content='details of attention and all the other steps in a transformer. Also, on my second channel I just posted a talk I gave a couple months ago about this topic for the company TNG in Munich. Sometimes I actually prefer the content I make as a casual talk rather than a produced video, but I leave it up to you which one of these feels like the better follow-on.')

In [20]:
vector_Store=FAISS.from_documents(chunnks,embeddings)

In [22]:
#retrieval step
retriever=vector_Store.as_retriever(search_type="similarity",search_kwargs={"k":4})

In [23]:
retriever.invoke("what is deep mind")

[Document(id='ecf00e63-6131-456d-a415-0afc1f0e00af', metadata={}, page_content="refined based on many example pieces of text. One of these training examples could be just a handful of words, or it could be thousands, but in either case, the way this works is to pass in all but the last word from that example into the model and compare the prediction that it makes with the true last word from the example. An algorithm called backpropagation is used to tweak all of the parameters in such a way that it makes the model a little more likely to choose the true last word and a little less likely to choose all the others. When you do this for many, many trillions of examples, not only does the model start to give more accurate predictions on the training data, but it also starts to make more reasonable predictions on text that it's never seen before. Given the huge number of parameters and the enormous amount of training data, the scale of computation involved in training a large language mode

In [24]:
#augmentation

llm=ChatGroq(model="openai/gpt-oss-120b")

prompt = PromptTemplate(
    template="""
You are a helpful assistant.
Answer ONLY from the provided transcript context.
If the context is insufficient, just say you don't know.

{context}
Question: {question}
""",
    input_variables=['context', 'question']
)

In [25]:
question = "is the topic of aliens discussed in this video? if yes then what was discussed"
retrieved_docs = retriever.invoke(question)

In [26]:
context_text="\n\n".join(doc.page_content for doc in retrieved_docs)
context_text

"details of attention and all the other steps in a transformer. Also, on my second channel I just posted a talk I gave a couple months ago about this topic for the company TNG in Munich. Sometimes I actually prefer the content I make as a casual talk rather than a produced video, but I leave it up to you which one of these feels like the better follow-on.\n\nImagine you happen across a short movie script that describes a scene between a person and their AI assistant. The script has what the person asks the AI, but the AI's response has been torn off. Suppose you also have this powerful magical machine that can take any text and provide a sensible prediction of what word comes next. You could then finish the script by feeding in what you have to the machine, seeing what it would predict to start the AI's answer, and then repeating this over and over with a growing script completing the dialogue. When you interact with a chatbot, this is exactly what's happening. A large language model i

In [27]:
final_prompt=prompt.invoke({"context":context_text,"question":question})
final_prompt

StringPromptValue(text="\nYou are a helpful assistant.\nAnswer ONLY from the provided transcript context.\nIf the context is insufficient, just say you don't know.\n\ndetails of attention and all the other steps in a transformer. Also, on my second channel I just posted a talk I gave a couple months ago about this topic for the company TNG in Munich. Sometimes I actually prefer the content I make as a casual talk rather than a produced video, but I leave it up to you which one of these feels like the better follow-on.\n\nImagine you happen across a short movie script that describes a scene between a person and their AI assistant. The script has what the person asks the AI, but the AI's response has been torn off. Suppose you also have this powerful magical machine that can take any text and provide a sensible prediction of what word comes next. You could then finish the script by feeding in what you have to the machine, seeing what it would predict to start the AI's answer, and then re

In [28]:
#generation
answer=llm.invoke(final_prompt)
print(answer)

content='No. The transcript provided talks about transformers, attention mechanisms, and large‑language‑model chatbots, but it makes no mention of aliens or any discussion of that topic.' additional_kwargs={'reasoning_content': 'We need to answer based only on provided transcript context. The transcript is about transformers, attention, large language models, a talk, etc. No mention of aliens. So answer: I don\'t know? Actually question: "is the topic of aliens discussed in this video? if yes then what was discussed". Since transcript doesn\'t mention aliens, we say no, it\'s not discussed. According to instruction: If context insufficient, say you don\'t know. But we have enough to know it\'s not discussed. So answer: No, aliens are not discussed.'} response_metadata={'token_usage': {'completion_tokens': 153, 'prompt_tokens': 587, 'total_tokens': 740, 'completion_time': 0.320014745, 'completion_tokens_details': {'reasoning_tokens': 109}, 'prompt_time': 0.026829988, 'prompt_tokens_deta

In [29]:
from langchain_core.runnables import RunnablePassthrough,RunnableParallel,RunnableLambda
from langchain_core.output_parsers import StrOutputParser

In [30]:
def format_text(retrieved_docs):
    context_text="\n\n".join(doc.page_content for doc in retrieved_docs)
    return context_text

In [31]:
parallel_chain=RunnableParallel({
    'context':retriever | RunnableLambda(format_text),
    'question':RunnablePassthrough()
})


In [32]:
parallel_chain.invoke("whjo is demis")
#

{'context': "[Submit subtitle corrections at criblate.com]\n\nImagine you happen across a short movie script that describes a scene between a person and their AI assistant. The script has what the person asks the AI, but the AI's response has been torn off. Suppose you also have this powerful magical machine that can take any text and provide a sensible prediction of what word comes next. You could then finish the script by feeding in what you have to the machine, seeing what it would predict to start the AI's answer, and then repeating this over and over with a growing script completing the dialogue. When you interact with a chatbot, this is exactly what's happening. A large language model is a sophisticated mathematical function that predicts what word comes next for any piece of text. Instead of predicting one word with certainty, though, what it does is assign a probability to all possible next words. To build a chatbot, what you do is lay out some text that describes an interactio

In [33]:
parser=StrOutputParser()

In [34]:
main_chain=parallel_chain | prompt | llm | parser

In [35]:
main_chain.invoke("summarize the entire video")

'**Video Summary**\n\nThe video explains how modern large‑language models (LLMs) work, focusing on the transformer architecture. It walks through the key components—especially the attention mechanism—and shows how each step transforms input tokens into contextual representations. The presenter emphasizes that, although researchers design the mathematical framework, the actual behavior of an LLM emerges from billions of parameters learned during training, making it hard to predict exactly why a particular word is chosen.\n\nKey points covered:\n\n1. **Transformers & Attention** – Visual and intuitive explanations of self‑attention, multi‑head attention, positional encoding, feed‑forward layers, and how these pieces fit together to process sequences.\n2. **Probability‑Based Word Prediction** – LLMs assign probabilities to every possible next token rather than picking a single “correct” word, which results in fluent and often useful continuations.\n3. **Emergent Behavior** – The model’s o